# 30. 用 AgentScope 在浏览器沙盒中打开 GitHub

## 这份 notebook 现在的角色

这份 notebook 现在**不再直接执行浏览器自动化**，而是作为这节课的学习说明文档。

真正执行浏览器操作的入口已经改成了项目根目录下的 Python 脚本：

- `browser_open_github.py`

这是因为在你的环境里：

- Windows
- Jupyter Notebook
- MCP stdio 子进程
- asyncio 事件循环

组合起来容易出现兼容问题。

所以我们采用更稳的工作流：

- notebook 负责学习说明
- `.py` 负责真实执行


## 为什么这次改成 `.py`

原因不是 AgentScope 不行，而是 notebook 运行时环境本身不够稳定。

对于 MCP 浏览器这类场景，`.py` 脚本通常更稳，因为：

- 它没有 notebook 自带的运行中事件循环干扰
- 更容易启动和管理外部子进程
- 更容易排查错误
- 更适合作为后续点击、输入、搜索的扩展基础

所以我们把职责分开：

### notebook 的职责

- 解释概念
- 说明代码结构
- 讲运行方法
- 讲如何观察输出

### `.py` 脚本的职责

- 连接 Playwright MCP
- 注册 AgentScope 浏览器工具
- 打开 GitHub
- 获取 snapshot
- 保存截图
- 正常关闭连接


## 执行脚本在哪里

脚本路径：

- `browser_open_github.py`

这是这节课真正的执行入口。

In [1]:
# 这一格只展示你应该运行的命令，不直接执行浏览器。
#
# 推荐在项目根目录的终端里运行：

command = "uv run python .\\browser_open_github.py"
print(command)


uv run python .\browser_open_github.py


## 这个脚本做了什么

脚本整体流程是：

1. 创建 `Toolkit`
2. 创建 `StdIOStatefulClient`
3. 通过 `npx @playwright/mcp@latest` 启动浏览器 MCP 服务
4. 把 MCP 暴露出来的浏览器工具注册进 `Toolkit`
5. 调用 `browser_navigate` 打开 `https://github.com`
6. 调用 `browser_snapshot` 获取页面文本快照
7. 调用 `browser_tabs` 查看当前标签页信息
8. 调用 `browser_take_screenshot` 保存截图
9. 关闭连接

从 AgentScope 的角度看，这里面最值得你记住的一点是：

**浏览器操作本质上也是工具调用。**

## 运行脚本时你应该重点观察什么

运行脚本后，请重点看这些输出：

### 1. 可用工具列表

这一步用来验证 AgentScope 是否成功拿到了浏览器工具，例如：

- `browser_navigate`
- `browser_snapshot`
- `browser_click`
- `browser_take_screenshot`

### 2. snapshot

这一步用来验证当前页面是不是已经变成 GitHub 首页。

### 3. tabs

这一步会告诉你当前标签页 URL 是否已经是 `https://github.com/`。

### 4. 截图路径

如果输出了 png 路径，并且文件存在，就说明这次浏览器操作已经成功完成。

## 你接下来应该怎么读 `browser_open_github.py`

建议按下面顺序理解：

1. 先看 `main()`，理解整体流程
2. 再看 `call_browser_tool()`，理解工具调用是怎么被包装的
3. 回头再看 `StdIOStatefulClient` 和 `Toolkit` 的创建

理解顺序不要反过来。

因为如果你一开始就陷入底层对象细节，容易看不清“整个脚本到底在干什么”。

## 下一步建议

等你把这个脚本跑通后，下一步可以继续往下扩展：

1. 点击页面元素
2. 输入文本
3. 做一次简单搜索
4. 最后再把这些工具交给 `ReActAgent` 自动调用

换句话说：

- 现在先学“手工控制工具”
- 再学“让 Agent 自动决定调用哪些工具”